# Clinical vision assistant

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zistgah/ghonchu/blob/main/notebooks/05_clinical_vision_assistant.ipynb)

For clinicians and medical educators: the assistant looks at instruments, vials, labels and layouts and says one sentence about what it sees. For teaching and practice only.

**Initial drop, 9 November 2026. Expect updates soon.** Extracted from the design discussion of the Ghonchu course (8 November 2026) and not yet run end to end; what was checked and what was changed is listed in [origin/SEED-DIFF.md](https://github.com/zistgah/ghonchu/blob/main/origin/SEED-DIFF.md).

Press ▶ on each step in order. Each step's code is folded behind its title; double-click a title to see it.

© 1993–2026 Abhishek Choudhary. All rights reserved. AyeAI. Licensed under GPL-3.0-or-later. AyeAI® and AyeAM® are registered trade marks of Abhishek Choudhary; AyeCNSe™ (Indian TM application No. 7415871, pending).

**Privacy and safety.** The camera picture travels from your phone's browser to this Colab runtime, which runs on Google's cloud, not on your phone; nothing is kept unless a cell saves it. Point the camera at the robot or the object, never at a person without their informed consent; children use it with an adult, and a parent's consent comes first. Do not show patient, client or personal information. These assistants teach; they are not medical, psychological or diagnostic devices, and no prompt makes anything compliant with GDPR, HIPAA or any other law.

Built from the shared vision tutor with this domain's prompt, question, panel, voice and timing exactly as discussed (14:31); the received cell differed from the shared one only in those, in its function names and in two panel colours, all recorded in origin/received.

In [ ]:
#@title Load the model, then watch and speak { display-mode: "form" }
# 1. Install required acceleration libraries for running 4-bit Vision Models on T4
!pip install -q unsloth
!pip install --upgrade xformers bitsandbytes accelerate transformers -q

import base64
import numpy as np
import cv2
import torch
from io import BytesIO
from PIL import Image
from IPython.display import display, Javascript
import google.colab.output
import IPython
from unsloth import FastVisionModel # Highly efficient memory engine for T4 VRAM

# 2. Load the Vision-Language Model in 4-bit (Uses ~7-8 GB VRAM, safely below T4 limits)
print("Loading Ghonchu Vision Brain (Qwen2.5-VL-7B 4-bit)...")
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
    load_in_4bit = True
)
FastVisionModel.for_inference(model) # Optimize model hooks for fast generation loops

# 3. Define the Global Pedagogical System Prompt for your Robotics Course
CLINICAL_SYSTEM_PROMPT = """
You are a precise, HIPAA-compliant Medical Vision Assistant.
You are watching a clinical space via a local workspace camera.
Analyze the provided visual frame under strict diagnostic parameters.
Identify the clinical instruments, medical vials, labels, or processes.
Point out any protocol anomalies, misalignments, or environment hazards.
Keep your response to ONE clear, direct sentence for audio conversion.
Do not mention patient names or corporate entities.
"""

def start_ghonchu_eye_tutor():
    js_code = Javascript('''
        async function runGhonchuTutor() {
            const container = document.createElement('div');
            container.style = 'padding: 16px; background: #0f172a; border-radius: 12px; color: #f1f5f9; font-family: sans-serif; max-width: 480px; margin: 12px auto; border: 2px solid #06b6d4; text-align: center;';
            
            const title = document.createElement('h3');
            title.textContent = '🩺 Local Clinical AI Vision Assistant';
            title.style = 'color: #06b6d4; margin-top: 0; font-size: 18px;';
            
            const status = document.createElement('p');
            status.id = 'ghonchu-status';
            status.textContent = '🔒 local loop running inside sandbox...';
            status.style = 'font-size: 14px; color: #94a3b8; font-weight: bold;';

            const video = document.createElement('video');
            video.style = 'width: 100%; border-radius: 6px; background: #020617; margin-bottom: 10px;';
            video.autoplay = true;
            video.playsInline = true;

            const stopBtn = document.createElement('button');
            stopBtn.textContent = 'Deactivate Loop';
            stopBtn.style = 'width: 100%; padding: 12px; background: #dc2626; color: white; border: none; border-radius: 6px; font-weight: bold; cursor: pointer;';

            container.appendChild(title);
            container.appendChild(status);
            container.appendChild(video);
            container.appendChild(stopBtn);
            document.body.appendChild(container);

            const synth = window.speechSynthesis;
            function speakFeedback(text) {
                if (synth.speaking) return;
                const utterance = new SpeechSynthesisUtterance(text);
                utterance.rate = 1.05;
                synth.speak(utterance);
            }

            const stream = await navigator.mediaDevices.getUserMedia({
                video: { facingMode: "environment" }
            }).catch(async (err) => {
                return await navigator.mediaDevices.getUserMedia({ video: true });
            });

            video.srcObject = stream;
            status.textContent = '👀 Assistant actively scanning workspace...';
            speakFeedback("Clinical observation matrix active. Place medical tools or layout items in line of sight.");

            const canvas = document.createElement('canvas');
            const ctx = canvas.getContext('2d');
            let activeLoop = true;

            stopBtn.onclick = () => {
                activeLoop = false;
                stream.getVideoTracks().forEach(track => track.stop());
                container.remove();
                speakFeedback("Observation sequence terminated safely.");
            };

            while (activeLoop) {
                if (video.videoWidth > 0) {
                    // Downscale image layout slightly to minimize processing delay matrices
                    canvas.width = 448; 
                    canvas.height = (video.videoHeight / video.videoWidth) * 448;
                    ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
                    
                    const frameData = canvas.toDataURL('image/jpeg', 0.75);
                    const result = await google.colab.kernel.invokeFunction('notebook.process_medical_frame', [frameData], {});
                    const jsonResponse = result.data['application/json'];
                    
                    if (jsonResponse && jsonResponse.voice_instruction) {
                        status.textContent = `🩺 Guidance: ${jsonResponse.voice_instruction}`;
                        speakFeedback(jsonResponse.voice_instruction);
                    }
                }
                // Processing locally via 4-bit VLM takes ~1.5 to 2 seconds per inference on a T4.
                // We check every 3 seconds to stay completely safe within memory cache pipelines.
                await new Promise(r => setTimeout(r, 3500)); 
            }
        }
        runGhonchuTutor();
    ''')
    display(js_code)

def process_ghonchu_frame(base64_frame):
    try:
        header, encoded = base64_frame.split(',')
        image_bytes = base64.b64decode(encoded)
        pil_image = Image.open(BytesIO(image_bytes)).convert("RGB")
        
        # 4. Formulate the Dynamic Context Chat Prompt targeting the local VLM layers
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": CLINICAL_SYSTEM_PROMPT},
                    {"type": "image"},
                    {"type": "text", "text": "Assess the medical tools, equipment layout, or documentation details shown. Identify anomalies."}
                ]
            }
        ]
        
        # Tokenize and pass variables directly to local GPU registers
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        inputs = tokenizer(pil_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
        
        with torch.inference_mode():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=45, 
                temperature=0.2,
                do_sample=True
            )
            
        # Extract the textual response slice cleanly
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Parse output down to clean assistant payload content string
        voice_instruction = generated_text.split("assistant\n")[-1].strip()
        
        return IPython.display.JSON({"voice_instruction": voice_instruction})
        
    except Exception as e:
        return IPython.display.JSON({"voice_instruction": ""})

# Bind context vectors
google.colab.output.register_callback('notebook.process_medical_frame', process_ghonchu_frame)

# Launch
start_ghonchu_eye_tutor()